# 📗 도메인 임베딩 파인튜닝: 우리 분야의 질의와 문서 연결하기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 환경:</strong> GPU VRAM 24GB(RTX 4090·3090)를 권장합니다. 입력 상한 512토큰·학습 배치 32 기준입니다.<br />
<strong style="font-size:1.13em;color:#1f3a5f;">CPU 실행:</strong> 가능하지만 학습이 느립니다. 시스템 RAM 16GB 이상·32GB를 권장합니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">도메인 임베딩 파인튜닝</strong>(domain embedding fine-tuning)은 우리 분야의 (질의, 정답 문서) 쌍으로 임베딩 모델을 이어서 학습하는 일입니다. 정답 문서를 오답 후보보다 높은 유사도로 찾도록 모델의 파라미터를 바꿉니다.</p></div>

<img src="images/20_domain_embedding.png" width="1050" alt="일상어 질의와 전문 용어 문서를 정답 쌍으로 연결해 임베딩 모델을 학습합니다.">

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">노무 상담 질문과 법령 조문은 같은 내용을 다른 표현으로 말합니다. 기존 상담 질문·근거 조문을 학습용 쌍 데이터셋으로 구성하고 검색 모델을 학습합니다. 같은 평가 질의로 학습 전·후를 비교한 뒤, 저장한 모델로 새로운 질문의 조문을 찾습니다.</p></div>

<img src="images/21_lesson01_goals.png" width="1050" alt="학습 목표: 1 학습 데이터 구축, 2 MNR 원리 이해, 3 모델 훈련 실행, 4 비교와 저장.">

**학습 목표** (그림의 번호와 같습니다)

1. 기존 질문·정답 조문으로 학습 데이터셋을 구축합니다.
2. 대조 학습과 MNR 손실에서 정답 쌍·배치 내 오답의 역할을 설명합니다.
3. (질의, 정답 문서) 데이터셋으로 `SentenceTransformerTrainer`를 사용해 파인튜닝합니다.
4. 같은 평가셋으로 학습 전·후를 비교하고, 저장한 모델로 새 질의를 검색합니다.

## 1. 도메인 학습 데이터셋 구축

### 1-1. 경로와 모델

`sentence-transformers`는 문장 임베딩 모델의 사용과 학습을 위한 파이썬 라이브러리입니다. 기존 실습과 같은 장치 선택 코드를 쓰고, 이번에는 `intfloat/multilingual-e5-small`을 학습합니다.

| 항목 | 이 실습의 모델 | 의미 |
|---|---|---|
| 모델 | `intfloat/multilingual-e5-small` | 한국어를 포함한 다국어 검색용 임베딩 모델 |
| 파라미터 수 | 약 1.18억 개(118M) | 학습으로 조정하는 모델 가중치의 전체 규모 |
| Transformer 층 | 12개 | 문맥을 처리하는 인코더 층 |
| 최대 입력 길이 | 512토큰 | 질의·문서 각각의 입력 상한. 프리픽스와 특수 토큰 포함 |
| 임베딩 차원 | 384 | 문장 하나를 표현하는 벡터의 길이 |

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">512는 글자 수가 아닌 토큰 수이며, 더 긴 입력은 상한에 맞춰 잘립니다. 토크나이저 예제의 <code style="font-size:0.95em;color:#345473;">max_length=512</code>에 해당하는 모델 입력 상한은 <code style="font-size:0.95em;color:#345473;">SentenceTransformer</code>의 <code style="font-size:0.95em;color:#345473;">max_seq_length</code>로 확인합니다. <a href="https://huggingface.co/intfloat/multilingual-e5-small">공식 모델 카드</a>와 <a href="https://huggingface.co/intfloat/multilingual-e5-small/blob/main/config.json">모델 설정</a>을 기준으로 합니다.</p></div>

이 모델은 질의에 `query: `, 문서에 `passage: `를 붙여 사용합니다. 학습과 검색에서 같은 형식을 쓰도록 두 프리픽스를 상수로 둡니다.

RunPod GPU 환경에서 진행하며, 준비 셀의 계산 장치가 `cuda`인지 확인합니다. 환경 구성은 기존 RunPod 부록을 따릅니다. 이 실습은 임베딩 API 키를 사용하지 않습니다.

In [ ]:
from pathlib import Path

import pandas as pd
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다 (지난 단원과 같은 코드)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 조문이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 이 단원의 임베딩 모델과 e5 계열 프리픽스(질의는 "query: ", 문서는 "passage: ")
MODEL_ID = "intfloat/multilingual-e5-small"
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "

### 1-2. 기존 질문과 정답 조문 읽기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">RAG 검색 평가 데이터셋처럼 질문과 정답 문서를 연결해 만듭니다(<code style="font-size:0.95em;color:#345473;">question</code>·<code style="font-size:0.95em;color:#345473;">relevant_doc_ids</code>를 연결했던 방식). 이 실습은 앞 단원의 기존 질문·정답 ID로 조문 본문을 연결한 자료를 그대로 사용합니다.</p></div>

1. 원조문을 검색 단위로 나누고 `doc_id`를 붙입니다.
2. 그 조문으로 답할 수 있는 사용자 질문을 직접 쓰거나 LLM으로 만듭니다. `positive`에는 RAG 답변 평가의 정답 답변이 아닌 **정답 조문 본문**을 연결합니다.
3. 질문이 원문의 조건으로 답되는지 검토하고, 같은 정답 조문에 연결된 질문이 그대로 반복되면 중복 행을 제거합니다. 여러 질문이 같은 조문을 정답으로 갖는 것은 유지합니다.
4. 같은 원 질문의 변형이 양쪽에 섞이지 않도록 그룹 단위로 학습·검증을 나눕니다.

| 파일 | 내용 | 필요한 키 |
|---|---|---|
| `labor_corpus.jsonl` | 검색할 조문 408개 | `doc_id`, `text` |
| `labor_train.jsonl` | 학습할 정답 쌍 663개 | `anchor`, `positive`, `doc_id` |
| `labor_eval.jsonl` | 검증용 정답 쌍 214개 | `anchor`, `positive`, `doc_id` |

학습·검증은 같은 세 열을 저장하되, 가중치 학습에는 `anchor`·`positive`를 쓰고 검색 평가에는 `anchor`·`doc_id`를 씁니다. 검증의 `positive`는 MNR 검증 손실 계산에도 씁니다. 검증용 쌍은 가중치 학습에 넣지 않으며, 검증 Recall@5로 최적 에폭을 선택합니다.

In [ ]:
# Windows에서도 한글 데이터를 같은 내용으로 읽도록 UTF-8을 명시합니다.
import json


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
# 검색 대상, 학습 쌍, 평가 질의는 역할이 달라 각각 읽습니다.
labor_corpus = load_jsonl(data_dir / "본실습" / "labor_corpus.jsonl")
labor_train = load_jsonl(data_dir / "본실습" / "labor_train.jsonl")
labor_eval = load_jsonl(data_dir / "본실습" / "labor_eval.jsonl")
print("조문:", len(labor_corpus), "| 학습 쌍:", len(labor_train), "| 평가 질의:", len(labor_eval))

In [ ]:
# 한 행에서 일상어 질의(anchor)와 정답 조문 본문(positive)의 연결을 읽습니다.
train_table = pd.DataFrame(labor_train)
display(train_table.head(1))

`anchor`는 검색창에 넣을 질의, `positive`는 그 질의의 정답 문서입니다. `doc_id`는 정답 조문을 식별하기 위한 값이며, 문장을 학습할 때 입력할 열은 `anchor`와 `positive` 두 개입니다.

### 1-3. 학습용 쌍 데이터셋 만들기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">이미 준비된 질문·정답 조문을 학습기가 받을 두 열의 <code style="font-size:0.95em;color:#345473;">Dataset</code>으로 구성합니다. 학습 파일의 질의에는 <code style="font-size:0.95em;color:#345473;">query: </code>, 정답 문서에는 <code style="font-size:0.95em;color:#345473;">passage: </code>를 한 번씩 붙이고, 평가 파일은 별도로 둡니다.</p></div>

이 단계의 결과는 학습용 `pair_ds`입니다. 다음 학습 전 검색 단계에서는 별도로 둔 평가 질의를 베이스 모델에 넣어, 나중에 학습 효과를 비교할 기준을 남깁니다.

In [ ]:
from datasets import Dataset

# 학습 입력은 질의(anchor), 정답 문서(positive) 두 열입니다. 식별용 doc_id는 넣지 않습니다.
pair_ds = Dataset.from_dict({
    "anchor": [QUERY_PREFIX + row["anchor"] for row in labor_train],
    "positive": [PASSAGE_PREFIX + row["positive"] for row in labor_train],
})
display(pair_ds)

## 2. 학습 전 검색 기준

### 2-1. 베이스 모델로 질의 하나 검색하기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">학습 데이터셋을 준비했으므로, 이제 학습 전 검색 품질을 기록합니다. 학습하기 전 모델을 <strong style="font-size:1.13em;color:#1f3a5f;">베이스 모델</strong>(base model)이라고 부르며, 이 모델은 비교를 위해 학습하지 않고 보관합니다.</p></div>

먼저 질문 하나로 무엇이 검색되는지 봅니다. `encode_texts`는 프리픽스를 붙인 문장을 정규화된 벡터로 만드는 공용 함수입니다.

In [ ]:
def encode_texts(model, texts, prefix):
    """텍스트 목록에 프리픽스를 붙여 정규화된 임베딩(numpy 배열)으로 만듭니다."""
    return model.encode(
        [prefix + text for text in texts],   # 프리픽스를 붙인 문장 목록
        normalize_embeddings=True,           # 길이 1로 정규화해 벡터끼리의 내적이 곧 코사인 유사도가 되게 합니다
        show_progress_bar=False,             # 진행 막대를 끕니다(평가 때마다 여러 번 불러 출력이 길어지지 않게)
    )

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer

# 이 객체는 학습하지 않고, 뒤에서 학습한 모델과 비교합니다.
base_model = SentenceTransformer(MODEL_ID, device=device)

# 공개 모델은 로그인 없이 내려받을 수 있습니다. 첫 실행에는 다운로드 시간이 필요합니다.
parameter_count = sum(parameter.numel() for parameter in base_model.parameters())
model_info_table = pd.DataFrame({
    "파라미터 수": [parameter_count],
    "입력 상한(토큰)": [base_model.max_seq_length],
    "임베딩 차원": [base_model.get_embedding_dimension()],
})
display(model_info_table)

,파라미터 수,입력 상한(토큰),임베딩 차원
0,117653760,512,384


In [ ]:
# 정규화한 벡터의 내적은 코사인 유사도와 같습니다.
doc_ids = [doc["doc_id"] for doc in labor_corpus]
base_doc_emb = encode_texts(base_model, [doc["text"] for doc in labor_corpus], PASSAGE_PREFIX)

In [ ]:
# 기존 평가셋의 질의 하나를 고릅니다. 이 질의는 학습 데이터셋에 넣지 않습니다.
example_query = "회사에서 지급받지 못한 임금을 3년간 청구하지 않았어도 지금 받을 수 있나요?"
query_texts = [row["anchor"] for row in labor_eval]
example_idx = query_texts.index(example_query)
example_doc_id = labor_eval[example_idx]["doc_id"]

# 질의 하나와 모든 조문의 유사도를 구하고, 높은 순으로 정렬합니다.
example_sims = encode_texts(base_model, [example_query], QUERY_PREFIX) @ base_doc_emb.T
example_order = np.argsort(-example_sims[0])

print("질의:", example_query)
print("정답 조문:", example_doc_id, "|", labor_corpus[doc_ids.index(example_doc_id)]["text"])

In [ ]:
# 상위 5개 조문을 원문과 함께 읽어 정답과 무엇이 다른지 봅니다.
example_rows = []
for rank, j in enumerate(example_order[:5], start=1):
    example_rows.append({
        "순위": rank,
        "조문": doc_ids[j],
        "정답 여부": doc_ids[j] == example_doc_id,
        "유사도": example_sims[0, j],
        "조문 내용": labor_corpus[j]["text"],
    })
example_table = pd.DataFrame(example_rows)
display(example_table.round(3))

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li><code style="font-size:0.95em;color:#345473;">정답 여부</code>와 조문 원문을 함께 봅니다. 정답 조문이 몇 위에 검색됐는지, 원문이 질문의 조건에 답하는지 확인합니다.</li>
</ul></div>

### 2-2. 평가셋으로 출발점 남기기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">검색 품질 평가 단원에서 배운 지표를 그대로 사용합니다. 이 평가 파일은 질의마다 정답 ID를 하나씩 기록했으므로, <strong style="font-size:1.13em;color:#1f3a5f;">Recall@5</strong>는 그 정답이 상위 5개에 든 질의의 비율입니다. Recall@1은 1위 정답 비율, MRR은 정답이 더 앞에 올수록 커지는 점수입니다.</p></div>

아래 함수는 정답 순위를 구한 뒤 세 지표를 계산합니다.

In [ ]:
import numpy as np


def gold_ranks(model, corpus, evalset, query_prefix=QUERY_PREFIX, passage_prefix=PASSAGE_PREFIX):
    """평가 질의마다 정답 문서가 유사도 순으로 몇 위인지(1위부터) 담은 배열을 돌려줍니다."""
    doc_ids = [doc["doc_id"] for doc in corpus]
    doc_emb = encode_texts(model, [doc["text"] for doc in corpus], passage_prefix)
    # 학습·평가 파일은 같은 세 열을 쓰며, 검색 평가는 질의와 정답 ID를 사용합니다.
    query_emb = encode_texts(model, [row["anchor"] for row in evalset], query_prefix)

    # 정규화한 벡터끼리의 내적이 코사인 유사도입니다. sims는 (질의 수, 문서 수) 행렬입니다
    sims = query_emb @ doc_emb.T

    # 질의마다 유사도가 높은 순으로 문서 id를 늘어놓고, 정답 문서가 몇 번째인지 셉니다
    ranks = []
    for i, row in enumerate(evalset):
        ranked_ids = [doc_ids[j] for j in np.argsort(-sims[i])]
        ranks.append(ranked_ids.index(row["doc_id"]) + 1)
    return np.array(ranks)


def retrieval_scores(ranks):
    """정답 순위 배열로 Recall@1, Recall@5, MRR을 계산합니다."""
    return {
        "Recall@1": round(float(np.mean(ranks <= 1)), 3),  # 정답 문서가 1위인 질의의 비율
        "Recall@5": round(float(np.mean(ranks <= 5)), 3),  # 정답 문서가 5위 안에 든 질의의 비율
        "MRR": round(float(np.mean(1 / ranks)), 3),        # 평균(1 / 정답 순위)
    }

In [12]:
# 학습에 쓰지 않은 평가 질의로 학습 전 기준 점수를 기록합니다.
base_ranks = gold_ranks(base_model, labor_corpus, labor_eval)
base_scores = pd.DataFrame([retrieval_scores(base_ranks)], index=["베이스 모델"])
display(base_scores)

,Recall@1,Recall@5,MRR
베이스 모델,0.603,0.78,0.68


### 2-3. 일반 분야 검색의 출발점

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">노무 검색이 좋아져도 다른 분야 검색은 나빠질 수 있으므로, <strong style="font-size:1.13em;color:#1f3a5f;">일반 분야 질문 83개</strong>의 학습 전 점수도 남깁니다. 기존 관광 43개·상식 30개·가상 도서관 10개 질문을 사용하며, <strong style="font-size:1.13em;color:#1f3a5f;">분야마다 자기 검색 문서 모음에서 평가한 순위</strong>를 합칩니다.</p></div>

상식 30개는 Day 70의 질문·정답 쌍을 가져와 질문을 검색어, 정답 문장을 검색 문서로 사용합니다. 예: “대한민국의 수도는 어디인가요?”를 검색해 “대한민국의 수도는 서울입니다.” 문서(`common:01`)를 찾습니다. 도서관 10개는 수업용 가상 규정입니다. 83개 검색 질의의 결과만으로 일반능력 전체의 변화를 단정하지 않습니다.

In [ ]:
# 학습에 쓰지 않는 일반 분야 평가 자료입니다. 검증 쌍의 세 열은 노무와 같습니다.
general_corpus = load_jsonl(data_dir / "본실습" / "general_corpus.jsonl")
general_eval = load_jsonl(data_dir / "본실습" / "general_eval.jsonl")
# 평가 쌍에는 분야 열이 없으므로 정답 doc_id를 문서의 분야와 연결합니다.
general_doc_domain = {doc["doc_id"]: doc["domain"] for doc in general_corpus}
general_domains = np.array([general_doc_domain[row["doc_id"]] for row in general_eval])
general_domain_names = ["관광", "상식", "도서관"]

# 분야별 후보 문서 수와 질의 수를 확인합니다.
general_doc_counts = {domain: sum(doc["domain"] == domain for doc in general_corpus) for domain in general_domain_names}
general_data_table = pd.DataFrame([
    {"분야": domain, "검색 문서 수": general_doc_counts[domain], "평가 질의 수": int(np.sum(general_domains == domain))}
    for domain in general_domain_names
])
display(general_data_table)

In [ ]:
# 관광·상식·도서관 질의는 각자의 분야 문서에서만 검색합니다.
base_general_ranks = np.empty(len(general_eval), dtype=int)
for domain in general_domain_names:
    domain_mask = general_domains == domain  # 현재 분야의 질의 위치만 True입니다.
    domain_corpus = [doc for doc in general_corpus if doc["domain"] == domain]
    domain_eval = [row for row, keep in zip(general_eval, domain_mask) if keep]
    base_general_ranks[domain_mask] = gold_ranks(base_model, domain_corpus, domain_eval)

# 분야별 후보 문서에서 구한 83질의 순위를 합쳐 학습 전 일반 검색 점수를 봅니다.
general_base_score_table = pd.DataFrame({"베이스 모델": retrieval_scores(base_general_ranks)}).T
general_base_score_table.index.name = "일반 83질의"
display(general_base_score_table)

## 3. 대조 학습과 MNR 손실

### 3-1. 정답 쌍과 배치 내 오답

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">대조 학습</strong>(contrastive learning)은 정답 문서는 더 가깝게, 오답 문서는 상대적으로 더 멀게 구분하도록 유사도를 조정하는 <strong style="font-size:1.13em;color:#1f3a5f;">학습 방식</strong>입니다. 이 실습에서는 질의와 정답 문서가 맞는 쌍입니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">MNR 손실</strong>(Multiple Negatives Ranking Loss)은 그 목적을 구현하는 대표 <strong style="font-size:1.13em;color:#1f3a5f;">손실 함수 중 하나</strong>입니다. 여러 (질의, 정답 문서) 쌍을 한 배치에 모으고, 각 질의가 자기 정답 문서를 고르도록 학습합니다. 다른 쌍의 정답 문서는 그 질의의 <strong style="font-size:1.13em;color:#1f3a5f;">배치 내 오답</strong>(in-batch negative) 후보가 됩니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">관계:</strong> 대조 학습(학습 방식) → MNR(손실 함수) → Softmax와 크로스 엔트로피(손실을 계산하는 방식).</p></div>

<img src="images/22_mnr_inbatch.png" width="1050" alt="두 정답 쌍을 한 배치에 놓으면 각 질의는 자기 정답과 다른 쌍의 문서를 비교합니다.">

**MNR 손실의 계산**

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">유사도 점수를 Softmax로 확률로 바꾸고, 정답 확률의 크로스 엔트로피를 배치에서 평균냅니다.</p></div>

$\displaystyle s_{ij}=20\cos(q_i,d_j)$

$\displaystyle p_i=\frac{\exp(s_{ii})}{\sum_{j=1}^{B}\exp(s_{ij})}$

$\displaystyle L_{\mathrm{MNR}}=\frac{1}{B}\sum_{i=1}^{B}-\log p_i$

- $B$: 현재 배치의 질문·정답 문서 쌍 수입니다.
- $q_i$: $i$번째 질의 벡터 하나, $d_j$: $j$번째 문서 벡터 하나입니다. $d_i$가 자기 정답입니다.
- $s_{ij}$: 질의 $i$와 문서 $j$의 점수입니다. `20`은 점수 차이를 조절하는 기본 `scale`입니다.
- $p_i$: 배치의 문서 후보 중 자기 정답에 준 확률 하나입니다. 검색 정확도와 같은 값은 아닙니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">손실의 해석:</strong> 정답 확률이 0.8이면 <code style="font-size:0.95em;color:#345473;">−log(0.8)≈0.223</code>, 0.2이면 <code style="font-size:0.95em;color:#345473;">−log(0.2)≈1.609</code>입니다. 정답에 낮은 확률을 줄수록 벌점이 커지므로, 학습은 정답을 다른 후보보다 높게 구분하는 방향으로 모델을 바꿉니다.</p></div>

오답을 따로 작성하지 않아도 다른 쌍의 문서를 후보로 사용합니다. 32쌍이면 정답 1개와 오답 후보 31개를 비교합니다. 후보 구성이 다른 학습의 손실 절댓값은 모델 품질 순위로 직접 비교하지 않으며, 검색 품질은 별도 검증 질문으로 확인합니다.

### 3-2. 같은 정답 문서가 두 번 들어오는 문제

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">서로 다른 질문이 같은 조문을 정답으로 가질 수 있습니다. 두 쌍을 같은 배치에 넣으면, 자기 정답과 같은 문서가 다른 쌍에서 온 오답 후보에도 나타납니다. 실제 정답을 오답으로 취급하는 것을 <strong style="font-size:1.13em;color:#1f3a5f;">거짓 오답</strong>(false negative)이라고 부릅니다.</p></div>

<img src="images/04_duplicate_false_negative.png" width="1050" alt="서로 다른 질의가 같은 정답 조문을 가지면, 배치에서 같은 조문이 정답과 오답 역할을 동시에 할 수 있습니다.">

`BatchSamplers.NO_DUPLICATES`는 질문·정답 중 하나라도 다른 행과 같은 문자열이면 다른 배치로 나눕니다. 문자열 중복 검사이며 의미가 비슷한 문장까지 판별하지는 못합니다. 그 문제는 교안 02에서 다룹니다.

## 4. 쌍 데이터로 파인튜닝

### 4-1. 학습할 모델과 손실

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">앞서 만든 <code style="font-size:0.95em;color:#345473;">pair_ds</code>로 학습할 모델을 준비합니다. 베이스 모델과 같은 출발점의 새 객체를 만들고, <code style="font-size:0.95em;color:#345473;">MultipleNegativesRankingLoss</code>에 이 학습용 모델을 넘깁니다.</p></div>

In [ ]:
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss

# base_model은 비교용으로 두고 새 모델의 파라미터만 바꿉니다.
tuned_model = SentenceTransformer(MODEL_ID, device=device)
loss = MultipleNegativesRankingLoss(tuned_model)

### 4-2. 검증과 학습 설정

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">배치 32·학습률 <code style="font-size:0.95em;color:#345473;">5e-5</code>로 최대 5에폭 학습합니다. 매 에폭 MNR 검증 손실과 전체 조문의 검색 Recall@5를 계산하고, <strong style="font-size:1.13em;color:#1f3a5f;">검증 Recall@5가 가장 높은 체크포인트</strong>를 복원합니다. Recall@5가 2회 연속 개선되지 않으면 일찍 종료합니다. 같은 Recall@5는 개선으로 세지 않으며, 동점이면 먼저 선택한 체크포인트를 유지합니다.</p></div>

검증셋은 가중치 학습에는 쓰지 않지만 최적 에폭 선택에 쓰므로 독립 테스트셋이 아닙니다. **일반 83질의는 모델 선택에 쓰지 않고**, 학습 후 다른 분야 검색의 변화만 점검합니다. [InformationRetrievalEvaluator](https://sbert.net/docs/package_reference/sentence_transformer/evaluation.html#informationretrievalevaluator)는 질의·전체 문서·정답 ID로 검색 지표를 계산합니다.

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator
from sentence_transformers.sentence_transformer.training_args import BatchSamplers
from sentence_transformers.util import cos_sim
from transformers import EarlyStoppingCallback

# 검증도 같은 두 열과 프리픽스를 사용하지만, 파라미터 갱신에는 넣지 않습니다.
eval_ds = Dataset.from_dict({
    "anchor": [QUERY_PREFIX + row["anchor"] for row in labor_eval],
    "positive": [PASSAGE_PREFIX + row["positive"] for row in labor_eval],
})
# 검증 질의마다 전체 문서에서 정답을 검색해 Recall@5를 계산합니다.
retrieval_evaluator = InformationRetrievalEvaluator(
    # 질의 ID는 정답 매핑과 연결할 문자열 키이고, 값은 프리픽스가 붙은 질문입니다.
    queries={str(i): QUERY_PREFIX + row["anchor"] for i, row in enumerate(labor_eval)},
    # 문서 ID -> 프리픽스가 붙은 검색 문서입니다.
    corpus={doc["doc_id"]: PASSAGE_PREFIX + doc["text"] for doc in labor_corpus},
    # 질의 ID -> 정답 조문 ID 집합입니다.
    relevant_docs={str(i): {row["doc_id"]} for i, row in enumerate(labor_eval)},
    name="labor",  # 지표 이름에 포함되어 eval_labor_cosine_recall@5를 만듭니다.
    score_functions={"cosine": cos_sim},  # 문서 점수는 코사인 유사도로 비교합니다.
    precision_recall_at_k=[1, 5],  # Recall@1과 Recall@5를 계산합니다.
    batch_size=64,  # 평가 문장을 64개씩 인코딩합니다.
    write_csv=False,
)

args = SentenceTransformerTrainingArguments(
    output_dir=str(output_dir / "labor_tuned_run"),
    num_train_epochs=5,                         # 최대 에폭 수. 개선이 없으면 그 전에 종료합니다.
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    warmup_steps=0.1,                           # 초반 10%의 갱신 동안 학습률을 서서히 올립니다.
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # 질문·정답 중 같은 문자열이 있으면 다른 배치로 나눕니다.
    eval_strategy="epoch",                    # 매 에폭 검증 손실과 검색 점수를 계산합니다.
    save_strategy="epoch",
    load_best_model_at_end=True,               # 종료 후 검증 Recall@5 최적 모델을 복원합니다.
    # evaluator의 name=labor, cosine 점수, Recall@5로 만들어진 지표 키입니다.
    metric_for_best_model="eval_labor_cosine_recall@5",
    greater_is_better=True,                    # Recall@5는 값이 클수록 좋은 모델입니다.
    save_total_limit=2,
    logging_steps=7,
    report_to="none",
    seed=42,                                    # 배치를 묶는 난수 시드를 고정합니다.
    use_cpu=(device.type == "cpu"),            # GPU(cuda)이면 False, CPU이면 True입니다.
)

In [ ]:
# 모델, 학습 설정, 데이터, 손실을 학습기 하나에 연결합니다.
trainer = SentenceTransformerTrainer(
    model=tuned_model,
    args=args,
    train_dataset=pair_ds,
    eval_dataset=eval_ds,
    evaluator=retrieval_evaluator,
    loss=loss,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

### 4-3. 학습 실행

`train()`은 MNR 손실이 작아지도록 파라미터를 갱신하고, 종료할 때 검증 Recall@5가 가장 높았던 모델을 `tuned_model`에 복원합니다. 아래 기록에서 검증 손실과 검색 점수의 변화가 항상 같은 방향인지는 확인하되, 선택 기준은 Recall@5입니다.

In [18]:
# global_step은 진행한 갱신 횟수, training_loss는 갱신 과정의 평균 손실입니다.
train_output = trainer.train()
display(train_output)

# 학습기가 남긴 매 에폭 검증 기록에서 핵심 지표만 골라 봅니다.
epoch_history = pd.DataFrame([
    {"에폭": log["epoch"], "검증 MNR 손실": round(log["eval_loss"], 4),
     "검증 Recall@5": round(log["eval_labor_cosine_recall@5"], 3)}
    for log in trainer.state.log_history if "eval_labor_cosine_recall@5" in log
])
display(epoch_history)

# 복원된 최적 체크포인트와 선택 점수를 확인합니다.
best_checkpoint_table = pd.DataFrame([{
    "선택 기준": "검증 Recall@5 최대", "최적 Recall@5": round(trainer.state.best_metric, 3),
    "복원 체크포인트": trainer.state.best_model_checkpoint,
}])
display(best_checkpoint_table)

Epoch,Training Loss,Validation Loss,Labor Cosine Accuracy@1,Labor Cosine Accuracy@3,Labor Cosine Accuracy@5,Labor Cosine Accuracy@10,Labor Cosine Precision@1,Labor Cosine Precision@5,Labor Cosine Recall@1,Labor Cosine Recall@5,Labor Cosine Ndcg@10,Labor Cosine Mrr@10,Labor Cosine Map@100
1,0.372476,0.319777,0.658879,0.822430,0.869159,0.934579,0.658879,0.173832,0.658879,0.869159,0.795763,0.751393,0.753536
2,0.108024,0.270416,0.626168,0.831776,0.906542,0.929907,0.626168,0.181308,0.626168,0.906542,0.789560,0.743113,0.745206
3,0.047456,0.250149,0.607477,0.859813,0.901869,0.920561,0.607477,0.180374,0.607477,0.901869,0.781770,0.735164,0.738342
4,0.047666,0.236157,0.621495,0.855140,0.915888,0.929907,0.621495,0.183178,0.621495,0.915888,0.786699,0.739045,0.742016
5,0.031586,0.245206,0.612150,0.845794,0.915888,0.929907,0.612150,0.183178,0.612150,0.915888,0.785007,0.736740,0.739431


,에폭,검증 MNR 손실,검증 Recall@5
0,1.0,0.3198,0.869
1,2.0,0.2704,0.907
2,3.0,0.2501,0.902
3,4.0,0.2362,0.916
4,5.0,0.2452,0.916


,선택 기준,최적 Recall@5,복원 체크포인트
0,검증 Recall@5 최대,0.916,../output/labor_tuned_run/checkpoint-84


## 5. 학습 전·후 비교

### 5-1. 같은 평가셋의 점수

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">학습한 모델에 같은 조문과 같은 평가 질의를 줍니다. 바꾼 것은 모델뿐이므로, 두 행의 차이가 이번 학습 후 검색 결과에 나타난 변화입니다.</p></div>

이 평가셋은 최적 에폭 선택에도 사용한 검증셋이므로, 아래 점수는 독립 테스트 성능이 아닙니다.

In [19]:
# 노무 평가의 비교 기준이 될 베이스 순위를 계산합니다.
tuned_ranks = gold_ranks(tuned_model, labor_corpus, labor_eval)

# 핵심 비교는 출발점과 쌍 학습 모델 두 개입니다.
score_table = pd.DataFrame(
    [retrieval_scores(base_ranks), retrieval_scores(tuned_ranks)],
    index=["베이스 모델", "쌍 학습 모델"],
)
display(score_table)

,Recall@1,Recall@5,MRR
베이스 모델,0.603,0.780,0.680
쌍 학습 모델,0.621,0.916,0.742


<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li><code style="font-size:0.95em;color:#345473;">베이스 모델</code> 행이 학습 전 기준입니다. 쌍 학습 모델 행과 같은 지표를 비교합니다.</li>
</ul></div>

### 5-2. 처음 검색한 질의의 변화

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">전체 점수와 함께, 학습 전에 보았던 같은 질문의 정답 순위도 나란히 봅니다.</p></div>

In [20]:
# 질의와 정답은 고정하고 모델에 따른 순위만 비교합니다.
example_rank_table = pd.DataFrame({
    "모델": ["베이스 모델", "쌍 학습 모델"],
    "정답 조문": [example_doc_id, example_doc_id],
    "정답 순위": [int(base_ranks[example_idx]), int(tuned_ranks[example_idx])],
})
print("질의:", example_query)
display(example_rank_table)

질의: 회사에서 지급받지 못한 임금을 3년간 청구하지 않았어도 지금 받을 수 있나요?


,모델,정답 조문,정답 순위
0,베이스 모델,근로기준법 제49조,12
1,쌍 학습 모델,근로기준법 제49조,2


<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li>정답 순위 숫자가 작아지면 이 질의에서 정답 조문이 더 앞에 온 것입니다. 전체 질의의 변화는 앞선 점수표와 따로 봅니다.</li>
</ul></div>

### 5-3. 일반 분야 검색의 퇴행과 개선폭

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">학습 전과 <strong style="font-size:1.13em;color:#1f3a5f;">같은 일반 질문·정답·후보 문서</strong>로 쌍 학습 모델을 평가합니다. <code style="font-size:0.95em;color:#345473;">Recall@5 변화(pp)</code>는 베이스 대비 퍼센트포인트 차이이며 음수이면 이 표본의 검색 성능이 하락한 것입니다.</p></div>

In [ ]:
# 학습 전과 같은 분야별 후보 집합에서 쌍 학습 모델의 정답 순위를 구합니다.
tuned_general_ranks = np.empty(len(general_eval), dtype=int)
for domain in general_domain_names:
    domain_mask = general_domains == domain  # 현재 분야의 질의 위치만 True입니다.
    domain_corpus = [doc for doc in general_corpus if doc["domain"] == domain]
    domain_eval = [row for row, keep in zip(general_eval, domain_mask) if keep]
    tuned_general_ranks[domain_mask] = gold_ranks(tuned_model, domain_corpus, domain_eval)

In [22]:
# 전체·분야별 평가 범위를 준비합니다.
general_groups = {"전체(일반 분야)": np.full(len(general_eval), True)}  # True 83개로 전체 질의를 선택합니다.
general_groups.update({domain: general_domains == domain for domain in general_domain_names})  # 같은 분야 위치만 True입니다.

# 일반 평가 점수 표에 사용할 모델별 순위를 정리합니다.
general_model_ranks = {"베이스 모델": base_general_ranks, "쌍 학습 모델": tuned_general_ranks}

# 각 평가 범위에서 점수와 베이스 대비 변화량을 계산합니다.
general_rows = []
for group, mask in general_groups.items():
    before_r5 = float(np.mean(base_general_ranks[mask] <= 5))  # 이 묶음의 베이스 비율이며 반올림 전 값입니다.
    for name, ranks in general_model_ranks.items():
        scores = retrieval_scores(ranks[mask])  # mask가 True인 질의의 순위만 평가합니다.
        general_rows.append({
            "평가 묶음": group, "모델": name,
            "문서 수": len(general_corpus) if group == "전체(일반 분야)" else general_doc_counts[group],
            "질의 수": int(mask.sum()),  # True인 위치의 개수입니다.
            **scores,  # Recall@1·Recall@5·MRR 세 열을 합칩니다.
            "Recall@5 변화(pp)": round(100 * (float(np.mean(ranks[mask] <= 5)) - before_r5), 2),  # 비율 차이에 100을 곱합니다.
        })

# 점수와 변화량을 한 테이블로 표시합니다.
general_score_table = pd.DataFrame(general_rows)
display(general_score_table)

,평가 묶음,모델,문서 수,질의 수,Recall@1,Recall@5,MRR,Recall@5 변화(pp)
0,전체(일반 분야),베이스 모델,433,83,0.699,0.855,0.766,0.00
1,전체(일반 분야),쌍 학습 모델,433,83,0.723,0.831,0.775,-2.41
2,관광,베이스 모델,388,43,0.442,0.721,0.566,0.00
3,관광,쌍 학습 모델,388,43,0.488,0.674,0.578,-4.65
4,상식,베이스 모델,30,30,1.000,1.000,1.000,0.00
5,상식,쌍 학습 모델,30,30,1.000,1.000,1.000,0.00
6,도서관,베이스 모델,15,10,0.900,1.000,0.925,0.00
7,도서관,쌍 학습 모델,15,10,0.900,1.000,0.950,0.00


In [23]:
# 노무 개선과 일반 검색 변화를 같은 눈금으로 비교합니다.
change_rows = []
for group, before, after in [("노무", base_ranks, tuned_ranks), ("일반 83질의", base_general_ranks, tuned_general_ranks)]:
    before_r5 = float(np.mean(before <= 5))  # 반올림 전 비율로 변화량을 계산합니다.
    after_r5 = float(np.mean(after <= 5))
    change_rows.append({"평가 분야": group, "베이스 Recall@5": round(before_r5, 3), "쌍 학습 Recall@5": round(after_r5, 3), "변화(pp)": round(100 * (after_r5 - before_r5), 2)})
regression_change_table = pd.DataFrame(change_rows)
display(regression_change_table)

,평가 분야,베이스 Recall@5,쌍 학습 Recall@5,변화(pp)
0,노무,0.780,0.916,13.55
1,일반 83질의,0.855,0.831,-2.41


<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li><code style="font-size:0.95em;color:#345473;">변화(pp)</code>는 베이스 대비 퍼센트포인트 차이입니다. 음수이면 상위 5개 정답 질문이 줄고, 양수이면 늘며, 0이면 같습니다.</li>
<li>도서관은 10질의라 정답 순위가 한 건 바뀌면 Recall@5가 10pp 변합니다. 전체 점수와 분야별 점수를 함께 봅니다.</li>
</ul></div>

## 6. 저장한 모델로 새 질의 검색

### 6-1. 저장하고 다시 불러오기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">학습한 모델을 <code style="font-size:0.95em;color:#345473;">output/labor_tuned</code>에 저장합니다. 이 폴더를 교안 02와 선택 모델 비교 실험에서도 불러옵니다.</p></div>

In [ ]:
# 검증 Recall@5가 가장 높아 복원된 모델과 토크나이저를 저장합니다.
tuned_model.save_pretrained(str(output_dir / "labor_tuned"))

In [ ]:
# 모델 이름 대신 저장 폴더를 전달하면 같은 방식으로 다시 사용할 수 있습니다.
loaded_model = SentenceTransformer(str(output_dir / "labor_tuned"), device=device)

### 6-2. 새 질문으로 상위 조문 찾기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">학습·평가 파일에 없는 질문을 넣습니다. 질의와 문서의 벡터를 모두 같은 모델로 만들어야 유사도를 비교할 수 있으므로, 문서 벡터도 불러온 모델로 다시 만듭니다.</p></div>

In [26]:
# 기존 실험의 새 질의입니다. 저장한 모델에서도 프리픽스를 똑같이 붙입니다.
new_query = "그만둔 지 3주가 지났는데 회사가 아직 퇴직금을 안 줬어요. 이렇게 늦게 줘도 되나요?"
search_doc_emb = encode_texts(loaded_model, [doc["text"] for doc in labor_corpus], PASSAGE_PREFIX)
search_sims = encode_texts(loaded_model, [new_query], QUERY_PREFIX) @ search_doc_emb.T

# 상위 조문은 제목만 보지 않고 원문의 조건까지 대조합니다.
search_rows = []
for rank, j in enumerate(np.argsort(-search_sims[0])[:5], start=1):
    search_rows.append({
        "순위": rank, "조문": doc_ids[j], "유사도": search_sims[0, j],
        "조문 내용": labor_corpus[j]["text"],
    })
search_table = pd.DataFrame(search_rows)
display(search_table.round(3))

,순위,조문,유사도,조문 내용
0,1,퇴직급여법 제10조,0.595,제10조(퇴직금의 시효) 이 법에 따른 퇴직금을 받을 권리는 3년간 행사하지 아니하면 시효로 인하여 소멸한다.
1,2,퇴직급여법 제8조 제1항,0.470,제8조(퇴직금제도의 설정 등) ① 퇴직금제도를 설정하려는 사용자는 계속근로기간 1년에 대하여 30일분 이상의 평균임금을 퇴직금으로 퇴직 근로자에게 지급할 수 있는 제도를 설정하여야 한다.
2,3,퇴직급여법 제9조 제1항,0.462,"제9조(퇴직금의 지급 등) ① 사용자는 근로자가 퇴직한 경우에는 그 지급사유가 발생한 날부터 14일 이내에 퇴직금을 지급하여야 한다. 다만, 특별한 사정이 있는 경우에는 당사자 간의 합의에 따라 지급기일을 연장할 수 있다. <개정 2021.4.13>"
3,4,근로기준법 제49조,0.426,제49조(임금의 시효) 이 법에 따른 임금채권은 3년간 행사하지 아니하면 시효로 소멸한다.
4,5,퇴직급여법 제12조 제2항,0.407,"제12조(퇴직급여등의 우선변제) ② 제1항에도 불구하고 최종 3년간의 퇴직급여등은 사용자의 총재산에 대하여 질권 또는 저당권에 의하여 담보된 채권, 조세ㆍ공과금 및 다른 채권에 우선하여 변제되어야 한다."


<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li>질문은 퇴직 뒤 지급 시점을 묻습니다. 상위 조문의 원문이 지급 기한을 다루는지 확인합니다. 유사도 점수만으로 법률적 정답을 판정할 수는 없습니다.</li>
</ul></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">모델을 검색기에 연결할 때</strong></p>
<ul>
<li>기존 검색기의 임베딩 모델 자리에 저장 폴더를 사용합니다. 질의·문서 프리픽스도 그대로 유지합니다.</li>
<li>임베딩 모델을 바꾸거나 다시 학습하면 기존 문서 벡터 전체를 새 모델로 다시 만듭니다. 새 문서만 추가할 때는 현재 모델로 그 문서의 벡터를 만들어 추가합니다.</li>
</ul></div>

## 7. 🖐️ 함께 따라하기: 증상으로 질환 문서 찾기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">같은 쌍 학습 과정을 의학 데이터에 적용해 일상어 증상 질의와 질환 문서를 연결합니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul>
<li><strong style="font-size:1.13em;color:#1f3a5f;">데이터:</strong> <a href="https://ko.wikipedia.org">한국어 위키백과</a>의 질환 문서 99개입니다(<a href="https://creativecommons.org/licenses/by-sa/4.0/deed.ko">CC BY-SA 4.0</a>). 학습 쌍은 195개, 평가 질의는 130개입니다.</li>
<li><strong style="font-size:1.13em;color:#1f3a5f;">표현:</strong> 질의는 증상을 일상어로 쓴 검색 문장이고, 문서는 의학 용어로 쓰여 있습니다. 이 실습은 문서 검색이며 의학적 진단이나 조언이 아닙니다.</li>
<li><strong style="font-size:1.13em;color:#1f3a5f;">키:</strong> 문서는 <code style="font-size:0.95em;color:#345473;">doc_id</code>, <code style="font-size:0.95em;color:#345473;">disease</code>, <code style="font-size:0.95em;color:#345473;">text</code>를 사용하며 학습·평가 파일의 키는 본실습과 같습니다.</li>
</ul></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> <code style="font-size:0.95em;color:#345473;">...</code>만 주석에 맞춰 채웁니다. 완성 코드는 <code style="font-size:0.95em;color:#345473;">정답/교안_01_정답.ipynb</code>의 같은 절에 있습니다. 위 준비·공용 함수·베이스 모델 셀을 실행한 상태에서 시작합니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정</strong></p>
<ul>
<li>최대 5에폭, 배치 16, 학습률 <code style="font-size:0.95em;color:#345473;">5e-5</code>, <code style="font-size:0.95em;color:#345473;">logging_steps=5</code>입니다.</li>
<li>매 에폭 검증·저장, 검증 Recall@5 최적 복원, 2회 연속 미개선 시 종료는 본실습과 같습니다.</li>
<li>교안 02가 불러올 수 있도록 <code style="font-size:0.95em;color:#345473;">output/med_tuned</code>에 저장합니다.</li>
</ul></div>

### 7-1. 데이터 읽기

In [ ]:
# 문서와 학습·평가 질의의 역할은 노무 데이터와 같습니다.
med_corpus = load_jsonl(data_dir / "따라하기" / "med_corpus.jsonl")
med_train = load_jsonl(data_dir / "따라하기" / "med_train.jsonl")
med_eval = load_jsonl(data_dir / "따라하기" / "med_eval.jsonl")
print("질환 문서:", len(med_corpus), "| 학습 쌍:", len(med_train), "| 평가 질의:", len(med_eval))

In [ ]:
# 일상어 질의와 의학 문서의 정답 연결을 한 행으로 읽습니다.
med_train_table = pd.DataFrame(med_train)
display(med_train_table.head(1))

### 7-2. 쌍 데이터셋과 손실

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">질의·문서 프리픽스와 MNR 손실을 본실습처럼 연결합니다.</p></div>

In [ ]:
# 🖐️ 함께 따라하기: 질의에는 QUERY_PREFIX, 정답 문서에는 PASSAGE_PREFIX를 붙여 두 열을 채우세요.
med_pair_ds = Dataset.from_dict({
    "anchor": ...,
    "positive": ...,
})
display(med_pair_ds)

In [ ]:
# 노무 학습 모델 대신 같은 베이스 이름에서 의학용 모델을 새로 시작합니다.
med_tuned_model = SentenceTransformer(MODEL_ID, device=device)

# 🖐️ 함께 따라하기: 학습할 의학 모델을 넘겨 MNR 손실을 만드세요.
med_loss = ...

### 7-3. 학습하고 저장하기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">의학 검증 쌍으로 MNR 검증 손실을, 질환 문서 전체에서 검색 Recall@5를 계산합니다. 같은 검증 질의로 최적 에폭을 선택하므로 독립 테스트 성능은 아닙니다. 최적 모델을 교안 02에서 불러올 수 있도록 저장합니다.</p></div>

In [ ]:
# 의학 검증 쌍도 질의·정답 두 열로 만들고 프리픽스를 한 번씩 붙입니다.
med_eval_ds = Dataset.from_dict({
    "anchor": [QUERY_PREFIX + row["anchor"] for row in med_eval],
    "positive": [PASSAGE_PREFIX + row["positive"] for row in med_eval],
})
# 검증 질의마다 전체 문서에서 정답을 검색해 Recall@5를 계산합니다.
med_evaluator = InformationRetrievalEvaluator(
    # 질의 ID는 정답 매핑과 연결할 문자열 키이고, 값은 프리픽스가 붙은 질문입니다.
    queries={str(i): QUERY_PREFIX + row["anchor"] for i, row in enumerate(med_eval)},
    # 문서 ID -> 프리픽스가 붙은 검색 문서입니다.
    corpus={doc["doc_id"]: PASSAGE_PREFIX + doc["text"] for doc in med_corpus},
    # 질의 ID -> 정답 조문 ID 집합입니다.
    relevant_docs={str(i): {row["doc_id"]} for i, row in enumerate(med_eval)},
    name="med",  # 지표 이름에 포함되어 eval_med_cosine_recall@5를 만듭니다.
    score_functions={"cosine": cos_sim},  # 문서 점수는 코사인 유사도로 비교합니다.
    precision_recall_at_k=[1, 5],  # Recall@1과 Recall@5를 계산합니다.
    batch_size=64,  # 평가 문장을 64개씩 인코딩합니다.
    write_csv=False,
)

# 🖐️ 함께 따라하기: 위 설정에 맞는 에폭 수와 배치 크기를 채우세요.
med_args = SentenceTransformerTrainingArguments(
    output_dir=str(output_dir / "med_tuned_run"),
    num_train_epochs=...,
    per_device_train_batch_size=...,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    warmup_steps=0.1,                           # 초반 학습률을 서서히 올리는 비율입니다.
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # 질문·정답 중 같은 문자열이 있으면 다른 배치로 나눕니다.
    eval_strategy="epoch",                    # 매 에폭 검증 손실과 검색 점수를 계산합니다.
    save_strategy="epoch",
    load_best_model_at_end=True,               # 종료 후 검증 Recall@5 최적 모델을 복원합니다.
    # evaluator의 name=med, cosine 점수, Recall@5로 만들어진 지표 키입니다.
    metric_for_best_model="eval_med_cosine_recall@5",
    greater_is_better=True,                    # Recall@5는 값이 클수록 좋은 모델입니다.
    save_total_limit=2,
    logging_steps=5,
    report_to="none",
    seed=42,                                    # 배치를 묶는 난수 시드를 고정합니다.
    use_cpu=(device.type == "cpu"),             # GPU(cuda)이면 False, CPU이면 True입니다.
)

In [ ]:
# 🖐️ 함께 따라하기: 의학 모델·학습/검증 데이터·evaluator·손실·조기 종료 콜백을 학습기로 묶으세요.
med_trainer = ...

med_output = med_trainer.train()
display(med_output)

# 매 에폭 검증 기록에서 손실과 검색 점수를 비교합니다.
med_epoch_history = pd.DataFrame([
    {"에폭": log["epoch"], "검증 MNR 손실": round(log["eval_loss"], 4),
     "검증 Recall@5": round(log["eval_med_cosine_recall@5"], 3)}
    for log in med_trainer.state.log_history if "eval_med_cosine_recall@5" in log
])
display(med_epoch_history)

# 복원된 최적 체크포인트와 선택 점수를 확인합니다.
med_best_checkpoint_table = pd.DataFrame([{
    "선택 기준": "검증 Recall@5 최대", "최적 Recall@5": round(med_trainer.state.best_metric, 3),
    "복원 체크포인트": med_trainer.state.best_model_checkpoint,
}])
display(med_best_checkpoint_table)

In [ ]:
# 검증 Recall@5가 가장 높아 복원된 의학 모델을 교안 02에서 불러옵니다.
med_tuned_model.save_pretrained(str(output_dir / "med_tuned"))

### 7-4. 학습 전·후 평가

의학 평가 질의를 고정하고 모델만 바꾸어 비교합니다.

In [ ]:
# 베이스 모델은 노무·의학 어느 데이터로도 학습하지 않은 출발점입니다.
med_base_ranks = gold_ranks(base_model, med_corpus, med_eval)

# 🖐️ 함께 따라하기: 의학 문서·평가 질의로 med_tuned_model의 정답 순위를 구하세요.
med_tuned_ranks = ...

med_score_table = pd.DataFrame(
    [retrieval_scores(med_base_ranks), retrieval_scores(med_tuned_ranks)],
    index=["베이스 모델", "쌍 학습 모델"],
)
display(med_score_table)

### 7-5. 다른 분야 검색의 변화

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">의학 검증</strong>은 가중치 학습에 넣지 않은 의학 질문으로 같은 분야의 일반화를 점검하고 최적 에폭을 선택합니다. <strong style="font-size:1.13em;color:#1f3a5f;">다른 분야 퇴행 점검</strong>은 의학 학습 전·후에 관광·상식·도서관 검색 성능이 유지되는지 확인하는 별도 평가입니다.</p></div>

앞에서 사용한 일반 83질의와 베이스 순위를 재사용합니다. 분야마다 자기 후보 문서에서 검색하고, 계산한 순위를 원래 질의 순서에 넣습니다.

In [ ]:
# 베이스 모델의 일반 검색 순위는 앞에서 계산한 값을 그대로 씁니다.
med_general_model_ranks = {"베이스 모델": base_general_ranks}
med_general_ranks = np.empty(len(general_eval), dtype=int)
for domain in general_domain_names:
    domain_mask = general_domains == domain  # 현재 분야의 질의 위치만 True입니다.
    domain_corpus = [doc for doc in general_corpus if doc["domain"] == domain]
    domain_eval = [row for row, keep in zip(general_eval, domain_mask) if keep]
    # 🖐️ 함께 따라하기: 의학 쌍 학습 모델로 분야 문서·질의의 정답 순위를 구하세요.
    med_general_ranks[domain_mask] = ...
med_general_model_ranks["의학 쌍 학습 모델"] = med_general_ranks

In [36]:
# 전체 83질의와 각 분야의 점수·베이스 대비 변화를 함께 봅니다.
# 앞에서 만든 전체·분야별 범위와 모델별 순위로 점수·변화량을 계산합니다.
med_general_rows = []
for group, mask in general_groups.items():
    before_r5 = float(np.mean(base_general_ranks[mask] <= 5))  # 이 묶음의 베이스 비율이며 반올림 전 값입니다.
    for name, ranks in med_general_model_ranks.items():
        scores = retrieval_scores(ranks[mask])  # mask가 True인 질의의 순위만 평가합니다.
        med_general_rows.append({
            "평가 묶음": group, "모델": name,
            "문서 수": len(general_corpus) if group == "전체(일반 분야)" else general_doc_counts[group],
            "질의 수": int(mask.sum()),  # True인 위치의 개수입니다.
            **scores,  # Recall@1·Recall@5·MRR 세 열을 합칩니다.
            "Recall@5 변화(pp)": round(100 * (float(np.mean(ranks[mask] <= 5)) - before_r5), 2),  # 비율 차이에 100을 곱합니다.
        })

# 점수와 변화량을 한 테이블로 표시합니다.
med_general_score_table = pd.DataFrame(med_general_rows)
display(med_general_score_table)

,평가 묶음,모델,문서 수,질의 수,Recall@1,Recall@5,MRR,Recall@5 변화(pp)
0,전체(일반 분야),베이스 모델,433,83,0.699,0.855,0.766,0.0
1,전체(일반 분야),의학 쌍 학습 모델,433,83,0.735,0.855,0.795,0.0
2,관광,베이스 모델,388,43,0.442,0.721,0.566,0.0
3,관광,의학 쌍 학습 모델,388,43,0.512,0.721,0.620,0.0
4,상식,베이스 모델,30,30,1.000,1.000,1.000,0.0
5,상식,의학 쌍 학습 모델,30,30,1.000,1.000,1.000,0.0
6,도서관,베이스 모델,15,10,0.900,1.000,0.925,0.0
7,도서관,의학 쌍 학습 모델,15,10,0.900,1.000,0.933,0.0


<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul>
<li><code style="font-size:0.95em;color:#345473;">Recall@5 변화(pp)</code>는 베이스 대비 퍼센트포인트 차이입니다. 음수이면 상위 5개 정답 질문이 줄고, 양수이면 늘며, 0이면 같습니다. 전체 83질의와 분야별 점수를 나눠 읽습니다.</li>
</ul></div>

## 8. 핵심 정리

- **도메인 임베딩 파인튜닝:** 분야의 질의와 정답 문서로 모델의 파라미터를 갱신해, 표현이 달라도 정답 문서를 찾도록 학습합니다.
- **학습 데이터 구축:** 기존 질문·정답 조문에서 학습할 두 열을 만들고 질의·문서 프리픽스를 한 번씩 붙입니다. 평가 질의는 학습에 넣지 않고, 학습 전·후의 비교 기준으로 둡니다.
- **MNR 손실:** 자기 정답 문서를 다른 쌍의 문서보다 높은 유사도로 구분하도록 학습합니다. 쌍 32개라면 질의마다 정답 1개와 배치 내 오답 후보 31개를 비교합니다.
- **거짓 오답과 배치:** 같은 정답 문서가 중복되는 것은 `NO_DUPLICATES`로 막습니다. 문서의 내용까지 같은 답인지 확인하는 문제는 별도 검토가 필요합니다.
- **최적 에폭:** 최대 5에폭 중 검증 Recall@5가 가장 높은 모델을 복원합니다. 2회 연속 미개선이면 종료하며, 검증 손실 최저 모델을 고르는 설정은 아닙니다.
- **학습 효과와 퇴행:** 노무·의학의 모델 선택용 검증 점수와 일반 분야의 검색 변화를 구분합니다. 일반 83질의는 최적 에폭 선택에 사용하지 않습니다.
- **저장과 검색:** `labor_tuned`와 `med_tuned`에 저장합니다. 문서와 질의에 같은 모델을 사용하고, 모델을 바꾸면 문서 벡터도 다시 만듭니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 02에서는 MNR 손실의 실제 계산, 어려운 오답 수집, 거짓 오답 검토와 일반화·퇴행 평가를 다룹니다. 공개 모델과의 비교는 선택 실험인 <code style="font-size:0.95em;color:#345473;">부록_임베딩_모델_비교.ipynb</code>에서 이어갑니다.</p></div>